In [1]:
import requests
import pandas as pd

def actualizar_datos_f1():
    print("recolectando datos desde OpenF1.org...")
    
    # 1. URL de la API (Puedes cambiar los parámetros aquí)
    # URL telemetria
    url = "https://api.openf1.org/v1/car_data?driver_number=55&session_key=9159"
    
    try:
        response = requests.get(url)
        if response.status_code == 200:
            data = response.json()
            df = pd.DataFrame(data)
            
            print(f"✅ Datos descargados: {len(df)} registros encontrados.")
            
            # 2. TRANSFORMACIÓN (Limpieza automática)
            # Convertimos fechas
            if 'date' in df.columns:
                df['date'] = pd.to_datetime(df['date'], errors='coerce')
            
            # Rellenamos nulos de sensores (telemetría)
            df = df.ffill().bfill()
            
            # 3. EXPORTACIÓN (El CSV que leerá Power BI)
            df.to_csv('f1_car_data_limpio.csv', index=False, encoding='utf-8')
            
            print("🚀 ARCHIVO 'f1_car_data_limpio.csv' ACTUALIZADO Y LIMPIO.")
            return df
        else:
            print(f"❌ Error en la API: Código {response.status_code}")
    except Exception as e:
        print(f"❌ Error en la conexión: {e}")

# --- ESTE ES EL 'BOTÓN' ---
# Cada vez que corra esta línea, Python irá a la API y refrescará el CSV
df_actualizado = actualizar_datos_f1()

recolectando datos desde OpenF1.org...
✅ Datos descargados: 18470 registros encontrados.
🚀 ARCHIVO 'f1_car_data_limpio.csv' ACTUALIZADO Y LIMPIO.


In [2]:
import requests
import pandas as pd
import time

def super_actualizador_f1():
    print("🏎️ INICIANDO EXTRACCIÓN MASIVA DESDE OPENF1...")
    
    # Configuración de URLs (Puedo ajustar session_key o driver_number aquí)
    session_id = "9159" # Ejemplo: GP de Bélgica 2023
    driver_id = "55"   # Carlos Sainz
    
    urls = {
        "car_data_limpio": f"https://api.openf1.org/v1/car_data?driver_number={driver_id}&session_key={session_id}",
        "location_limpio": f"https://api.openf1.org/v1/location?driver_number={driver_id}&session_key={session_id}",
        "weather_limpio": f"https://api.openf1.org/v1/weather?session_key={session_id}",
        "stints_limpio": f"https://api.openf1.org/v1/stints?session_key={session_id}",
        "sessions_limpio": f"https://api.openf1.org/v1/sessions?session_key={session_id}"
    }
    
    for nombre_archivo, url in urls.items():
        try:
            print(f"📡 Descargando {nombre_archivo}...")
            response = requests.get(url)
            
            if response.status_code == 200:
                data = response.json()
                if not data:
                    print(f"⚠️ No hay datos nuevos para {nombre_archivo}.")
                    continue
                
                df = pd.DataFrame(data)
                
                # --- TRANSFORMACIÓN AUTOMÁTICA ---
                # 1. Convertir fechas (si existen)
                for col in df.columns:
                    if 'date' in col.lower() or 'time' in col.lower():
                        df[col] = pd.to_datetime(df[col], errors='coerce')
                
                # 2. Limpieza de nulos (Forward fill y Backward fill)
                df = df.ffill().bfill()
                
                # 3. Exportación a CSV
                df.to_csv(f'{nombre_archivo}.csv', index=False, encoding='utf-8')
                print(f"✅ {nombre_archivo}.csv actualizado con {len(df)} filas.")
                
            else:
                print(f"❌ Error en {nombre_archivo}: Código {response.status_code}")
            
            # Pausa de seguridad para no saturar la API
            time.sleep(0.5)
            
        except Exception as e:
            print(f"❌ Error crítico en {nombre_archivo}: {e}")

    print("\n🏆 ¡PROCESO COMPLETADO! Archivos listos para Power BI.")

# --- EJECUTAR EL BOTÓN ACTUALIZAR ---
super_actualizador_f1()

🏎️ INICIANDO EXTRACCIÓN MASIVA DESDE OPENF1...
📡 Descargando car_data_limpio...
✅ car_data_limpio.csv actualizado con 18470 filas.
📡 Descargando location_limpio...
✅ location_limpio.csv actualizado con 19041 filas.
📡 Descargando weather_limpio...
✅ weather_limpio.csv actualizado con 83 filas.
📡 Descargando stints_limpio...
✅ stints_limpio.csv actualizado con 73 filas.
📡 Descargando sessions_limpio...
✅ sessions_limpio.csv actualizado con 1 filas.

🏆 ¡PROCESO COMPLETADO! Archivos listos para Power BI.
